In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

sys.path.append(str(SRC_DIR))



In [15]:
from load_data import(
    load_hpo
)

from build_triples import(
    load_filtered_hpo_annotations,
    build_disease_hpo_triples,
    build_entity_metadata,
    save_triples,
    save_entity_metadata,
    get_triple_statistics,
)

from train_transe_pykeen import(
    create_triples_factory,
    split_triples_factory,
    train_transe_model,
    extract_disease_embeddings,
    save_embedding,
    save_metrics,
)

In [3]:
HPO_PATH = RAW_DIR / "hp.obo"
HPOA_FILTERED_PATH = PROCESSED_DIR / "hpoa_filtered.csv"

hpo = load_hpo(HPO_PATH)
hpoa_filtered = load_filtered_hpo_annotations(HPOA_FILTERED_PATH)

print("HPO nodes:", hpo.number_of_nodes())
print("HPO edges:", hpo.number_of_edges())
print("Filtered annotation rows:", len(hpoa_filtered))
print("Filtered diseases:", hpoa_filtered["database_id"].nunique())
print("Filtered HPO terms:", hpoa_filtered["hpo_id"].nunique())

hpoa_filtered.head()

HPO nodes: 19389
HPO edges: 23677
Filtered annotation rows: 20178
Filtered diseases: 1000
Filtered HPO terms: 4569


,database_id,disease_name,qualifier,hpo_id,reference,evidence,onset,frequency,sex,modifier,aspect,biocuration
0,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011097,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
1,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0002187,PMID:31675180,PCS,NaN,1/1,NaN,NaN,P,HPO:probinson[2021-06-21]
2,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0001518,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
3,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0032792,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
4,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011451,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]


In [4]:
triples = build_disease_hpo_triples(
    hpoa_filtered=hpoa_filtered,
    hpo=hpo,
    include_hpo_hierarchy=True,
    include_only_relevant_hpo_edges=True,
)

triple_stats = get_triple_statistics(triples)

triple_stats

{'triples_total': 31521,
 'entities_total': 11760,
 'relations_total': 2,
 'relation_has_phenotype': 20132,
 'relation_is_a': 11389}

In [5]:
triples.head(20)

,head,relation,tail
0,OMIM:619340,has_phenotype,HP:0011097
1,OMIM:619340,has_phenotype,HP:0002187
2,OMIM:619340,has_phenotype,HP:0001518
3,OMIM:619340,has_phenotype,HP:0032792
4,OMIM:619340,has_phenotype,HP:0011451
5,OMIM:619340,has_phenotype,HP:0010851
6,OMIM:619340,has_phenotype,HP:0001789
7,OMIM:619340,has_phenotype,HP:0200134
8,OMIM:619340,has_phenotype,HP:0002643
9,OMIM:619426,has_phenotype,HP:0000286


In [6]:
triples["relation"].value_counts()

relation
has_phenotype    20132
is_a             11389
Name: count, dtype: int64

In [7]:
entity_metadata = build_entity_metadata(
    hpoa_filtered=hpoa_filtered,
    hpo=hpo,
    triples=triples,
)

print("Entities:", len(entity_metadata))
print(entity_metadata["entity_type"].value_counts())

entity_metadata.head(20)

Entities: 11760
entity_type
phenotype    10760
disease       1000
Name: count, dtype: int64


,entity_id,entity_type,label
0,DECIPHER:21,disease,Miller-Dieker syndrome (MDS)
1,DECIPHER:3,disease,Williams-Beuren Syndrome (WBS)
2,DECIPHER:45,disease,Xq28 (MECP2) duplication
3,DECIPHER:54,disease,Angelman syndrome (Type 2)
4,DECIPHER:81,disease,15q26 overgrowth syndrome
5,HP:0000002,phenotype,Abnormality of body height
6,HP:0000003,phenotype,Multicystic kidney dysplasia
7,HP:0000008,phenotype,Abnormal morphology of female internal genitalia
8,HP:0000009,phenotype,Functional abnormality of the bladder
9,HP:0000010,phenotype,Recurrent urinary tract infections


In [8]:
TRIPLES_PATH = PROCESSED_DIR / "triples_disease_hpo.tsv"
ENTITY_METADATA_PATH = PROCESSED_DIR / "entity_metadata_disease_hpo.csv"

save_triples(triples, TRIPLES_PATH)
save_entity_metadata(entity_metadata, ENTITY_METADATA_PATH)

print("Saved triples to:", TRIPLES_PATH)
print("Saved metadata to:", ENTITY_METADATA_PATH)

Saved triples to: /workspaces/GraphRepresentationLearning/data/processed/triples_disease_hpo.tsv
Saved metadata to: /workspaces/GraphRepresentationLearning/data/processed/entity_metadata_disease_hpo.csv


In [9]:
triples_factory = create_triples_factory(
    triples=triples,
    create_inverse_triples=True,
)

print("Entities:", triples_factory.num_entities)
print("Relations:", triples_factory.num_relations)
print("Triples:", triples_factory.num_triples)

Entities: 11760
Relations: 4
Triples: 31521


In [10]:
training, validation, testing = split_triples_factory(
    triples_factory=triples_factory,
    ratios=(0.8, 0.1, 0.1),
    random_state=5,
)

print("Training triples:", training.num_triples)
print("Validation triples:", validation.num_triples)
print("Testing triples:", testing.num_triples)

Training triples: 25216
Validation triples: 3152
Testing triples: 3153


In [11]:
result = train_transe_model(
    training=training,
    validation=validation,
    testing=testing,
    embedding_dim=32,
    num_epochs=50,
    batch_size=256,
    learing_rate=0.001,
    random_seed=5,
    model_output_directory=MODELS_DIR / "transe_disease_hpo"
)

INFO:pykeen.triples.triples_factory:Creating inverse triples.
/workspaces/GraphRepresentationLearning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Training epochs on cpu: 100%|██████████| 50/50 [01:09<00:00,  1.39s/epoch, loss=0.0219, prev_loss=0.023] 
Evaluating on cpu:   0%|          | 0.00/3.15k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 3.15k/3.15k [00:15<00:00, 199triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 15.94s seconds
INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=11760, num_relations=4, create_inverse_triples=True, num_triples=25216) to fil

In [12]:
metrics = result.metric_results.to_flat_dict()

for key, value in list(metrics.items())[:20]:
    print(key, value)

head.optimistic.z_geometric_mean_rank 52.479304339593114
tail.optimistic.z_geometric_mean_rank 48.80056146587273
both.optimistic.z_geometric_mean_rank 72.04202857974116
head.realistic.z_geometric_mean_rank 52.479291712249335
tail.realistic.z_geometric_mean_rank 48.8005584008009
both.realistic.z_geometric_mean_rank 72.04201285111579
head.pessimistic.z_geometric_mean_rank 52.479279631570456
tail.pessimistic.z_geometric_mean_rank 48.80055619757842
both.pessimistic.z_geometric_mean_rank 72.04200142938086
head.optimistic.z_inverse_harmonic_mean_rank 55.114942671130784
tail.optimistic.z_inverse_harmonic_mean_rank 97.45383220336466
both.optimistic.z_inverse_harmonic_mean_rank 107.88222391621088
head.realistic.z_inverse_harmonic_mean_rank 55.11479838631354
tail.realistic.z_inverse_harmonic_mean_rank 97.45383493291145
both.realistic.z_inverse_harmonic_mean_rank 107.88212695217881
head.pessimistic.z_inverse_harmonic_mean_rank 55.114651280456606
tail.pessimistic.z_inverse_harmonic_mean_rank 97.45

In [13]:
METRICS_PATH = PROCESSED_DIR / "pykeen_transe_metrics.csv"

metrics_df = save_metrics(
    result=result,
    output_path=METRICS_PATH,
)

metrics_df.head(10)

,metric,value
0,head.optimistic.z_geometric_mean_rank,52.479304
1,tail.optimistic.z_geometric_mean_rank,48.800561
2,both.optimistic.z_geometric_mean_rank,72.042029
3,head.realistic.z_geometric_mean_rank,52.479292
4,tail.realistic.z_geometric_mean_rank,48.800558
5,both.realistic.z_geometric_mean_rank,72.042013
6,head.pessimistic.z_geometric_mean_rank,52.479280
7,tail.pessimistic.z_geometric_mean_rank,48.800556
8,both.pessimistic.z_geometric_mean_rank,72.042001
9,head.optimistic.z_inverse_harmonic_mean_rank,55.114943


In [16]:
disease_embeddings_transe = extract_disease_embeddings(
    result=result,
    entitiy_metadata=entity_metadata,
)

print("Disease embeddings:", disease_embeddings_transe.shape)
disease_embeddings_transe.head()

Disease embeddings: (1000, 35)


,entity_id,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,...,dim_24,dim_25,dim_26,dim_27,dim_28,dim_29,dim_30,dim_31,entity_type,label
0,DECIPHER:21,0.042787,0.248684,-0.203058,-0.046804,-0.283181,-0.067965,-0.023819,-0.003923,0.136725,...,-0.326444,-0.131847,0.220800,-0.042644,0.154087,-0.094088,0.079420,0.322761,disease,Miller-Dieker syndrome (MDS)
1,DECIPHER:3,0.087997,-0.206632,-0.120176,-0.352266,-0.101608,-0.221718,-0.040925,0.071674,0.411519,...,-0.250823,0.044286,-0.218914,-0.237703,0.038430,-0.260760,-0.184132,0.061635,disease,Williams-Beuren Syndrome (WBS)
2,DECIPHER:45,-0.086892,0.143391,-0.149582,-0.118088,-0.122956,-0.098960,0.071973,-0.390183,0.161120,...,-0.196338,0.023438,0.173035,-0.129244,0.025186,0.032387,-0.069286,-0.120627,disease,Xq28 (MECP2) duplication
3,DECIPHER:54,-0.145245,0.049770,-0.195604,-0.392991,-0.124197,0.341346,-0.086276,0.176357,0.059678,...,0.132165,0.159629,0.146437,0.251457,0.079966,-0.143056,-0.310975,-0.025079,disease,Angelman syndrome (Type 2)
4,DECIPHER:81,-0.244729,0.034498,-0.122347,-0.170408,-0.101192,-0.014939,-0.052999,-0.087859,0.007928,...,-0.321677,0.197775,0.062999,-0.087581,0.103390,-0.009194,-0.207620,0.375099,disease,15q26 overgrowth syndrome


In [17]:
TRANSE_EMBEDDINGS_PATH = PROCESSED_DIR / "disease_embeddings_transe.csv"

save_embedding(
    embeddings=disease_embeddings_transe,
    output_path=TRANSE_EMBEDDINGS_PATH,
)

print("Saved", TRANSE_EMBEDDINGS_PATH)

Saved /workspaces/GraphRepresentationLearning/data/processed/disease_embeddings_transe.csv


In [18]:
embedding_columns = [
    col for col in disease_embeddings_transe.columns
    if col.startswith("dim_")
]

print("Number of diseases:", disease_embeddings_transe["entity_id"].nunique())
print("Embedding dimensions:", len(embedding_columns))
print("Missing values:", disease_embeddings_transe[embedding_columns].isna().sum().sum())

disease_embeddings_transe[["entity_id", "label", "entity_type"]].head(10)

Number of diseases: 1000
Embedding dimensions: 32
Missing values: 0


,entity_id,label,entity_type
0,DECIPHER:21,Miller-Dieker syndrome (MDS),disease
1,DECIPHER:3,Williams-Beuren Syndrome (WBS),disease
2,DECIPHER:45,Xq28 (MECP2) duplication,disease
3,DECIPHER:54,Angelman syndrome (Type 2),disease
4,DECIPHER:81,15q26 overgrowth syndrome,disease
5,OMIM:100700,Achard syndrome,disease
6,OMIM:103420,"Alacrima, congenital",disease
7,OMIM:104300,Alzheimer disease,disease
8,OMIM:105300,Amyotrophic dystonic paraplegia,disease
9,OMIM:106210,Aniridia,disease
